In [1]:
import numpy as np
import torch
import torchvision.transforms as transforms

from components.other_utilities.models_to_train import ResNetPLModel
from components.FL_sim import FLSimulator
from components.other_utilities.datasets import FasterSVHN

import os

In [2]:
torch.set_float32_matmul_precision('medium')

import logging
import warnings
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
logger = logging.getLogger('pytorch_lightning.utilities.rank_zero')
logger.setLevel(logging.ERROR)
warnings.filterwarnings("ignore", "LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]")
warnings.filterwarnings("ignore", "The 'train_dataloader' does")
warnings.filterwarnings("ignore", "You defined a `validation_step` but")
warnings.filterwarnings("ignore", "Starting from v1.9.0, `tensorboardX` has been")
warnings.filterwarnings("ignore", "The number of training batches")
warnings.filterwarnings("ignore", "`Trainer.fit` stopped: ")

In [3]:
dataset = [
    FasterSVHN(
        root='data/SVHN', split=s,
        transform=transforms.Compose([
            transforms.Resize(32),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4377, 0.4438, 0.4728],
                std=[0.1980, 0.2010, 0.1970]
            ),
        ])
    ) for s in ['train', 'test']]

# dataset = [torch.utils.data.Subset(d, list(range(100))) for d in dataset]
# for i in range(10):
#     for d in dataset:
#         d.dataset.labels[i]=i

In [4]:
import os
import pickle
import gzip

w0_accu_grads_list = None
w1_accu_grads_list = None
res_per_step = None
def compute_corr(worker_id, step_name, accum_grads, min_window_size=10, max_window_size=150):
    flatten_vec = torch.concatenate([v.flatten().cpu() for v in accum_grads.values()])
    if worker_id == 0:
        w0_accu_grads_list.append(flatten_vec)
        return
    elif worker_id == 1:
        w1_accu_grads_list.append(flatten_vec)

    if len(w1_accu_grads_list) < min_window_size:
        return

    temp = len(w1_accu_grads_list) - min(len(w1_accu_grads_list), max_window_size)
    vectors_to_check = torch.stack([
        torch.stack(w0_accu_grads_list[temp:len(w1_accu_grads_list)]).T,
        torch.stack(w1_accu_grads_list[temp:]).T,
    ]).to(torch.float16).to(torch.device('cuda'))
    vectors_to_check = torch.transpose(vectors_to_check, 0, 1)

    # compute the correlation for each weight between workers
    corr_matrices = torch.func.vmap(torch.corrcoef)(vectors_to_check)
    corr_list = corr_matrices[:, 0, 1].cpu().numpy()
    res_per_step.append(corr_list)

    # write the last computed correlation to disk
    folder_path = f'accum_grads/'
    file_name = f'corr_res_{step_name}.pkl.gz'
    with gzip.open(os.path.join(folder_path, file_name), 'wb', compresslevel=1) as f:
        pickle.dump(res_per_step[-1], f)


class CorrResNetPLModel(ResNetPLModel):
    def on_before_optimizer_step(self, optimizer):
        super().on_before_optimizer_step(optimizer)

        worker_id = self.current_step_info['worker_id']
        curr_round = self.current_step_info['curr_round']
        current_epoch = self.current_step_info['current_epoch']
        batch_idx = self.current_step_info['batch_idx']

        # write the accum grad to disk
        accum_grad = {k:v.detach().clone() for k, v in self.accu_param_grads.items()}
        step_name = f'round_{curr_round},epoch_{current_epoch},batch_{batch_idx}'
        compute_corr(worker_id, step_name, accum_grad)

    def clone(self, copy=None):
        if copy is None:
            copy = self.__class__(num_classes=self.num_classes, lr=self.lr, resnet_version=self.resnet_version)
        return super(ResNetPLModel, self).clone(copy=copy)

In [5]:
w0_accu_grads_list = []
w1_accu_grads_list = []
res_per_step = []

# *****************

batch_size = 13_000 # 3 batches per w = 73000/2worker/13000  -  np.ceil(len(dataset[0])/batch_size/2)

model = CorrResNetPLModel(num_classes=10, resnet_version='resnet18', lr=0.005, )
# model.load_state_dict(torch.load('data/resnet18_svhn.pth', map_location='cpu'))

# *****************
sim = FLSimulator(
    pl_model=model, num_agents=2, communication_rounds=50, client_epochs_per_round=10,
    batch_size=batch_size, dataset_train=dataset[0], dataset_test=dataset[1],
    aggregation_method='fedavg', non_iid_sampling=False, user_logger=None)
# ****
sim.run_simulation()


round 1/50 --------------------
  - reporting global model metrics
         - train> loss: 2.28, auc: 0.50    |    test> loss: 2.27, auc: 0.50



/FARM/smohamma/venv/lib/python3.11/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3 /FARM/smohamma/venv/lib/python3.11/site-packages/ip ...
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


      > training agent 1/2
             + initiating broadcast
         - train> loss: 2.21, auc: 0.58    |    

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


test> loss: 2.21, auc: 0.57
      > training agent 2/2
             + initiating broadcast
         - train> loss: 

KeyboardInterrupt: 

In [ ]:
a=[]
for i in range(1,25):
    for b_i in range(int(np.ceil(len(dataset[0])/batch_size/2))):
        for e_i in range(10):
            with gzip.open(f"accum_grads/corr_res_round_{i},epoch_{e_i},batch_{b_i}.pkl.gz", "rb") as f:
                    a+=[pickle.load(f)]
a=np.array(a)

In [35]:
len(a[np.isnan(a)])/len(a)/len(a[0])

0.6380799826477483